In [5]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
import probeinterface

from probeinterface import Probe, ProbeGroup

import os
import numpy as np
from spikeinterface.core import concatenate_recordings

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import pearsonr
import pandas as pd
import numpy as np
from matplotlib.collections import LineCollection
from probeinterface import write_probeinterface, read_probeinterface
import spikeinterface.exporters as sexp
from spikeinterface.core import write_binary_recording
from pathlib import Path
import pickle
import MEArec as mr
from tabnanny import verbose
import spikeinterface as si
import numpy as np
from spikeinterface.core import get_template_extremum_channel
import scipy.spatial.distance
from scipy.sparse.csgraph import connected_components
import pickle
import pandas as pd
from utils_clique import (
    CliqueInfo,
    build_shank_cliques,
    neuron_inf_dict_to_dataframe,
    get_recording_clique,
    filter_neuron_inf_by_clique,
    filter_gt_detect_array_by_clique,
    prepare_training_data,
    train_autosort_model,
    build_sliding_cliques
)


In [2]:
recording, sorting = se.read_mearec("/media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s.h5")
probe = recording.get_probe()
recording_recorded = spre.bandpass_filter(recording, freq_min=300, freq_max=3000)
recording_recorded = spre.notch_filter(recording_recorded, freq=50)
recording_f = spre.common_reference(recording_recorded, reference="global", operator="median")

In [4]:
output_folder = '/media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s'
cliques = build_sliding_cliques(
    probe,
    clique_size=49,
    min_size=25,
    min_overlap=18,
    target_groups=12,
)

clique_info = {
    'cliques': cliques,  # List of CliqueInfo objects
    'clique_params': {
        'clique_size': 49,
        'min_size': 25,
        'min_overlap': 18,
        'target_groups': 12,
    },
    'probe_df': probe.to_dataframe(),  # Probe dataframe for verification
}

clique_info_path = f'{output_folder}/clique_info.pkl'
with open(clique_info_path, 'wb') as f:
    pickle.dump(clique_info, f)

[INFO] Built 12 cliques (target 12)
       Clique 00: channels 192-12 (49 channels)
       Clique 01: channels 103-115 (49 channels)
       Clique 02: channels 303-123 (49 channels)
       Clique 03: channels 23-227 (49 channels)
       Clique 04: channels 31-43 (49 channels)
       Clique 05: channels 326-338 (49 channels)
       Clique 06: channels 334-154 (49 channels)
       Clique 07: channels 54-258 (49 channels)
       Clique 08: channels 254-74 (49 channels)
       Clique 09: channels 165-177 (49 channels)
       Clique 10: channels 173-185 (49 channels)
       Clique 11: channels 371-383 (49 channels)


In [ ]:
# 将recording分割成6段，每段600s
total_duration_seconds = 3600  # 总时长3600秒
segment_duration_seconds = 600  # 每段600秒
n_segments = 6  # 总共6段

# 获取recording的采样率和总采样点数
sampling_frequency = recording_f.get_sampling_frequency()
total_num_samples = recording_f.get_num_samples()

# 计算每段的采样点数
segment_num_samples = int(segment_duration_seconds * sampling_frequency)

# 计算每个segment的采样点范围
segment_sample_ranges = {}  # {segment_idx: (start_sample, end_sample)}
segment_num_samples_dict = {}  # {segment_idx: num_samples}

for seg_idx in range(n_segments):
    start_sample = seg_idx * segment_num_samples
    # 最后一段可能不足600s，使用实际结束位置
    if seg_idx == n_segments - 1:
        end_sample = total_num_samples
    else:
        end_sample = (seg_idx + 1) * segment_num_samples
    
    segment_sample_ranges[seg_idx] = (start_sample, end_sample)
    segment_num_samples_dict[seg_idx] = end_sample - start_sample
    print(f"Segment {seg_idx}: 采样点范围 = [{start_sample}, {end_sample}), 采样点数 = {end_sample - start_sample}")

print(f"\n{'='*60}")
print(f"开始统一处理整个recording的sorting结果")
print(f"{'='*60}\n")

# 设置输出文件夹
combined_output_base = output_folder  # 使用之前定义的output_folder

# 统一的phy_folder路径（整个recording的sorting结果）
phy_folder = f'{combined_output_base}/phy_folder_for_kilosort'

# 读取整个recording的sorting结果
print("读取统一的sorting结果...")
sorting_curated_phy = se.read_phy(phy_folder, exclude_cluster_groups=["noise"])
print(f"读取到 {len(sorting_curated_phy.unit_ids)} 个units\n")

# 对整个recording创建analyzer
print("创建analyzer并计算extensions...")
analyzer_curated_phy = si.create_sorting_analyzer(
    sorting=sorting_curated_phy, 
    recording=recording_f,  # 使用整个recording
    format='binary_folder',
    folder=combined_output_base + '/analyzer_curated_temp',
    n_jobs=20, verbose = False
)

extensions_to_compute = [
    "random_spikes",
    "waveforms",
    "templates",
    "unit_locations",
    "template_similarity"
]

extension_params = {
    "unit_locations": {"method": "center_of_mass"},
    "template_similarity": {"method": "cosine_similarity"}
}

analyzer_curated_phy.compute(extensions_to_compute, extension_params=extension_params, n_jobs=20, verbose = False)
print("完成extensions计算\n")

# 获取neuron信息（整个recording）
templates_ext = analyzer_curated_phy.get_extension("templates")
templates_dense = templates_ext.data["average"]
sparsity = analyzer_curated_phy.sparsity
unit_locations_ext = analyzer_curated_phy.get_extension("unit_locations")
unit_locations = unit_locations_ext.get_data()
channel_locations = analyzer_curated_phy.get_channel_locations()

# 处理merge逻辑
if unit_locations.shape[1] >= 2:
    unit_distances = scipy.spatial.distance.cdist(
        unit_locations[:, :2], 
        unit_locations[:, :2], 
        metric="euclidean"
    )
else:
    unit_distances = scipy.spatial.distance.cdist(
        unit_locations, 
        unit_locations, 
        metric="euclidean"
    )

template_similarity_ext = analyzer_curated_phy.get_extension("template_similarity")
template_similarity = template_similarity_ext.get_data()

distance_threshold = 10.0
similarity_threshold = 0.95
num_units = len(analyzer_curated_phy.unit_ids)
pair_mask = np.zeros((num_units, num_units), dtype=bool)

for i in range(num_units):
    for j in range(i + 1, num_units):
        if unit_distances[i, j] < distance_threshold and template_similarity[i, j] > similarity_threshold:
            pair_mask[i, j] = True
            pair_mask[j, i] = True

n_components, labels = connected_components(
    csgraph=pair_mask, 
    directed=False, 
    return_labels=True
)

merge_unit_groups = []
unit_ids_list = analyzer_curated_phy.unit_ids
for component_id in range(n_components):
    unit_indices = np.where(labels == component_id)[0]
    if len(unit_indices) > 1:
        group = [unit_ids_list[i] for i in unit_indices]
        merge_unit_groups.append(group)

# 应用merge（如果有需要merge的units）
if len(merge_unit_groups) > 0:
    print(f"发现 {len(merge_unit_groups)} 组需要merge的units，开始merge...")
    analyzer_merged = analyzer_curated_phy.merge_units(
        merge_unit_groups=merge_unit_groups,
        censor_ms=0.3,
        merging_mode="hard",
        new_id_strategy="append",
        format='binary_folder',
        folder=combined_output_base + '/analyzer_merged',
        verbose=True,
        n_jobs=20
    )
    
    analyzer_merged.compute(extensions_to_compute, extension_params=extension_params, n_jobs=20, verbose = False)
    
    templates_ext_final = analyzer_merged.get_extension("templates")
    templates_dense_final = templates_ext_final.data["average"]
    sparsity_final = analyzer_merged.sparsity
    unit_locations_ext_final = analyzer_merged.get_extension("unit_locations")
    unit_locations_final = unit_locations_ext_final.get_data()
    channel_locations_final = analyzer_merged.get_channel_locations()
    sorting_final = analyzer_merged.sorting
    unit_ids_list_final = analyzer_merged.unit_ids
    
    # 生成position_waveforms
    position_waveforms_final = []
    for unit_id in unit_ids_list_final:
        unit_index = analyzer_merged.sorting.id_to_index(unit_id)
        template_dense_unit = templates_dense_final[unit_index, :, :]
        template_sparse_unit = sparsity_final.sparsify_waveforms(template_dense_unit[np.newaxis, :, :], unit_id)[0]
        sparse_channel_indices = sparsity_final.unit_id_to_channel_indices[unit_id]
        
        if len(sparse_channel_indices) == 0:
            position_waveform = np.zeros(templates_dense_final.shape[1], dtype=templates_dense_final.dtype)
            position_waveforms_final.append(position_waveform)
            continue
        
        sparse_channel_locations = channel_locations_final[sparse_channel_indices, :2]
        unit_location = unit_locations_final[unit_index, :2]
        
        distances = np.sqrt(np.sum((sparse_channel_locations - unit_location[np.newaxis, :])**2, axis=1))
        epsilon = 1e-10
        weights = 1.0 / (distances + epsilon)
        weights = weights / np.sum(weights)
        
        position_waveform = np.dot(template_sparse_unit, weights)
        position_waveforms_final.append(position_waveform)
    
    position_waveforms_final = np.array(position_waveforms_final)
    extremum_channels_final = get_template_extremum_channel(
        analyzer_merged, 
        peak_sign="neg",
        outputs="id"
    )
    
    channel_ids_list = list(analyzer_merged.recording.get_channel_ids())
else:
    print("无需merge units\n")
    # 不需要merge，使用原始结果
    templates_ext_final = analyzer_curated_phy.get_extension("templates")
    templates_dense_final = templates_ext_final.data["average"]
    sparsity_final = analyzer_curated_phy.sparsity
    unit_locations_ext_final = analyzer_curated_phy.get_extension("unit_locations")
    unit_locations_final = unit_locations_ext_final.get_data()
    channel_locations_final = analyzer_curated_phy.get_channel_locations()
    sorting_final = analyzer_curated_phy.sorting
    unit_ids_list_final = unit_ids_list
    
    # 生成position_waveforms
    position_waveforms_final = []
    for unit_id in unit_ids_list_final:
        unit_index = analyzer_curated_phy.sorting.id_to_index(unit_id)
        template_dense_unit = templates_dense_final[unit_index, :, :]
        template_sparse_unit = sparsity_final.sparsify_waveforms(template_dense_unit[np.newaxis, :, :], unit_id)[0]
        sparse_channel_indices = sparsity_final.unit_id_to_channel_indices[unit_id]
        
        if len(sparse_channel_indices) == 0:
            position_waveform = np.zeros(templates_dense_final.shape[1], dtype=templates_dense_final.dtype)
            position_waveforms_final.append(position_waveform)
            continue
        
        sparse_channel_locations = channel_locations_final[sparse_channel_indices, :2]
        unit_location = unit_locations_final[unit_index, :2]
        
        distances = np.sqrt(np.sum((sparse_channel_locations - unit_location[np.newaxis, :])**2, axis=1))
        epsilon = 1e-10
        weights = 1.0 / (distances + epsilon)
        weights = weights / np.sum(weights)
        
        position_waveform = np.dot(template_sparse_unit, weights)
        position_waveforms_final.append(position_waveform)
    
    position_waveforms_final = np.array(position_waveforms_final)
    extremum_channels_final = get_template_extremum_channel(
        analyzer_curated_phy, 
        peak_sign="neg",
        outputs="id"
    )
    
    channel_ids_list = list(analyzer_curated_phy.recording.get_channel_ids())

# 计算每个unit的channel_id（template中值不为0的通道）
print("计算每个unit的channel_id...")
channel_ids_dict = {}  # {unit_id: [channel_id1, channel_id2, ...]}
for idx, unit_id in enumerate(unit_ids_list_final):
    unit_index = sorting_final.id_to_index(unit_id)
    template_unit = templates_dense_final[unit_index, :, :]  # (n_samples, n_channels)
    
    # 找到template中值不为0的通道
    non_zero_channels = []
    for ch_idx in range(template_unit.shape[1]):  # 遍历channels（最后一个维度）
        if np.any(template_unit[:, ch_idx] != 0):  # 检查该通道在所有时间点的值
            # 将通道索引转换为通道名称（格式和extremum_channel一样）
            channel_name = str(channel_ids_list[ch_idx])
            non_zero_channels.append(channel_name)
    
    channel_ids_dict[unit_id] = non_zero_channels

print(f"完成channel_id计算，共处理{len(channel_ids_dict)}个units\n")

# 计算channel_snr（每个unit的各个channel的SNR）
print("计算channel_snr...")
n_channels = recording_f.get_num_channels()

# 计算noise_std（使用前10秒的数据）
duration_samples = int(10 * sampling_frequency)  # 10秒
max_samples = recording_f.get_num_samples()
actual_samples = min(duration_samples, max_samples)
traces = recording_f.get_traces(start_frame=0, end_frame=actual_samples)  # (n_timepoints, n_channels)

noise_std_detect = np.median(np.abs(traces) / 0.6745, axis=0)  # (n_channels,)

all_spike_times = []
all_spike_unit_ids = []
for unit_id in unit_ids_list_final:
    spike_train = sorting_final.get_unit_spike_train(unit_id)
    all_spike_times.extend(spike_train.tolist())
    all_spike_unit_ids.extend([unit_id] * len(spike_train))

n_spikes_total = len(all_spike_times)
n_spikes_sample = min(1000, n_spikes_total)
if n_spikes_sample > 0:
    random_indices = np.random.choice(n_spikes_total, size=n_spikes_sample, replace=False)
    sampled_spike_times = [all_spike_times[i] for i in random_indices]
    sampled_spike_unit_ids = [all_spike_unit_ids[i] for i in random_indices]
else:
    sampled_spike_times = []
    sampled_spike_unit_ids = []

# 提取这些spike的waveform并计算每个channel的负值amplitude
left_sample = 10
right_sample = 20
window_size = left_sample + right_sample

channel_snr_dict = {} 

for unit_id in unit_ids_list_final:
    channel_snr_dict[unit_id] = {}
    unit_spike_times = [st for st, uid in zip(sampled_spike_times, sampled_spike_unit_ids) if uid == unit_id]
    
    if len(unit_spike_times) == 0:
        unit_spike_times = sorting_final.get_unit_spike_train(unit_id).tolist()
        if len(unit_spike_times) > 1000:
            unit_spike_times = np.random.choice(unit_spike_times, size=1000, replace=False).tolist()
    
    unit_waveforms = []  # List of (n_channels, window_size)
    valid_spike_times = []
    
    for spike_time in unit_spike_times:
        start = spike_time - left_sample
        end = spike_time + right_sample

        waveform = recording_f.get_traces(start_frame=start, end_frame=end)  # (n_timepoints, n_channels)
        unit_waveforms.append(waveform)
        valid_spike_times.append(spike_time)
    
    if len(unit_waveforms) == 0:
        continue
    
    unit_waveforms = np.array(unit_waveforms)  # (n_spikes, n_timepoints, n_channels)
    
    spike_time_values = unit_waveforms[:, left_sample, :]  # (n_spikes, n_channels) - 每个spike在spike_time时刻各个channel的值
    
    channel_amplitudes = np.mean(spike_time_values, axis=0)  # (n_channels,) - 每个channel的平均值（在spike_time时刻）
    channel_snr = np.abs(channel_amplitudes) / noise_std_detect  # (n_channels,)
    
    # 只保存 channel_ids_dict[unit_id] 中列出的通道的 SNR
    unit_channel_ids = channel_ids_dict.get(unit_id, [])  # 获取该unit的channel_id列表
    
    for ch_idx, snr_value in enumerate(channel_snr):
        channel_id = str(channel_ids_list[ch_idx])
        # 只保存 channel_ids_dict 中列出的通道
        if channel_id in unit_channel_ids:
            channel_snr_dict[unit_id][channel_id] = float(snr_value)

print(f"完成channel_snr计算，共处理{len(channel_snr_dict)}个units\n")

# 生成整体的neuron_inf（所有units）
neuron_inf_all = {}
for idx, unit_id in enumerate(unit_ids_list_final):
    neuron_inf_all[unit_id] = {
        'location_x': float(unit_locations_final[idx, 0]),
        'location_y': float(unit_locations_final[idx, 1]),
        'position_waveform': position_waveforms_final[idx],
        'extremum_channel': extremum_channels_final[unit_id],
        'channel_id': channel_ids_dict[unit_id],
        'channel_snr': channel_snr_dict.get(unit_id, {})  # 添加channel_snr字段
    }

# 生成整体的gt_detect_array（所有spikes）
print("生成整体的gt_detect_array...")
spike_vector_final = sorting_final.to_spike_vector()
gt_detect_data_all = []

for spike in spike_vector_final:
    unit_index = spike['unit_index']
    unit_id = sorting_final.unit_ids[unit_index]
    sample_index = spike['sample_index']  # 全局采样点索引（跨所有segments）
    
    # 根据sample_index确定它属于哪个segment
    segment_index = None
    for seg_idx, (start_sample, end_sample) in segment_sample_ranges.items():
        if start_sample <= sample_index < end_sample:
            segment_index = seg_idx
            break
    
    if segment_index is None:
        # 如果无法确定segment，跳过（理论上不应该发生）
        continue
    
    # 计算segment内的相对采样点索引（精确到每个采样点）
    segment_start_sample, segment_end_sample = segment_sample_ranges[segment_index]
    relative_sample_index = sample_index - segment_start_sample
    
    time_seconds = relative_sample_index
    
    extremum_channel = extremum_channels_final[unit_id]
    
    gt_detect_data_all.append({
        'time': time_seconds,
        'unit_id': unit_id,
        'extremum_channel': str(extremum_channel),
        'segment_index': segment_index  # 保存segment_index用于精确筛选
    })

gt_detect_array_all = pd.DataFrame(gt_detect_data_all)
print(f"完成gt_detect_array生成，共{len(gt_detect_array_all)}个spikes\n")

# 获取每个clique的channel_ids集合（用于筛选）
print("准备按clique划分neuron和spikes...\n")

for clique in cliques:
    clique_id = clique.clique_id
    print(f"\n{'='*60}")
    print(f"处理 Clique {clique_id}")
    print(f"{'='*60}")
    
    clique_output_folder = f'{combined_output_base}/clique_{clique_id}'
    os.makedirs(clique_output_folder, exist_ok=True)
    
    # 获取该clique的channel_ids集合
    clique_channel_ids = set(clique.channel_ids)  # clique的channel_ids集合
    
    # 筛选属于该clique的neuron：neuron的channel_id全部在该clique内
    clique_unit_ids = []
    for unit_id in unit_ids_list_final:
        unit_channel_ids = set(channel_ids_dict[unit_id])  # 该unit的channel_ids集合
        # 如果unit的所有channel_id都在clique内，则属于该clique
        if len(unit_channel_ids) > 0 and unit_channel_ids.issubset(clique_channel_ids):
            clique_unit_ids.append(unit_id)
    
    print(f"筛选出 {len(clique_unit_ids)} 个属于该clique的units（共 {len(unit_ids_list_final)} 个units）")
    
    if len(clique_unit_ids) == 0:
        print(f"  Clique {clique_id} 没有符合条件的units，跳过")
        continue
    
    # 构建该clique的neuron_inf
    neuron_inf_clique = {unit_id: neuron_inf_all[unit_id] for unit_id in clique_unit_ids}
    
    # 筛选属于该clique的spikes（只保留属于clique_unit_ids的spikes）
    gt_detect_array_clique = gt_detect_array_all[gt_detect_array_all['unit_id'].isin(clique_unit_ids)].copy()
    
    print(f"筛选出 {len(gt_detect_array_clique)} 个属于该clique的spikes（共 {len(gt_detect_array_all)} 个spikes）")
    
    # 保存该clique的整体neuron_inf和gt_detect_array
    with open(clique_output_folder + '/neuron_inf.pickle', 'wb') as f:
        pickle.dump(neuron_inf_clique, f)
    
    gt_detect_array_clique.to_csv(clique_output_folder + '/gt_detect_array.csv', index=False)
    
    # 按segment分割该clique的数据
    for segment_idx in range(n_segments):
        print(f"\n处理 Segment {segment_idx} ({segment_idx * segment_duration_seconds}s - {(segment_idx + 1) * segment_duration_seconds}s)...")
        
        # 使用segment_index进行精确筛选
        mask = (gt_detect_array_clique['segment_index'] == segment_idx)
        spikes_in_segment = gt_detect_array_clique[mask].copy()
        
        # 删除segment_index列（因为已经筛选完成，不再需要）
        if 'segment_index' in spikes_in_segment.columns:
            spikes_in_segment = spikes_in_segment.drop(columns=['segment_index'])
        
        # 筛选neuron：计算每个neuron的firing rate，如果 < 0.5 Hz则删除
        neurons_in_segment = spikes_in_segment['unit_id'].unique()
        firing_rate_threshold = 0.5  # Hz
        valid_neurons = []
        removed_spikes_count = 0
        
        # 获取该segment的采样点数（用于计算firing rate）
        if segment_idx not in segment_num_samples_dict:
            print(f"  警告: Segment {segment_idx} 的采样点数未找到，跳过")
            continue
        segment_num_samples = segment_num_samples_dict[segment_idx]
        
        for unit_id in neurons_in_segment:
            if unit_id not in neuron_inf_clique:
                # 如果neuron不在neuron_inf_clique中，跳过
                continue
            
            # 计算该neuron在该segment中的spike数量
            neuron_spikes = spikes_in_segment[spikes_in_segment['unit_id'] == unit_id]
            spike_count = len(neuron_spikes)
            
            # 计算firing rate (spikes per second)，使用采样点数：spike_count * sampling_frequency / num_samples
            firing_rate = (spike_count * sampling_frequency) / segment_num_samples if segment_num_samples > 0 else 0
            
            if firing_rate >= firing_rate_threshold:
                valid_neurons.append(unit_id)
            else:
                # 删除该neuron的spikes
                removed_spikes_count += spike_count
                spikes_in_segment = spikes_in_segment[spikes_in_segment['unit_id'] != unit_id]
        
        # 只保留有效的neurons
        neuron_inf_segment = {unit_id: neuron_inf_clique[unit_id] for unit_id in valid_neurons}
        
        segment_output_folder = f'{clique_output_folder}/segment_{segment_idx}'
        os.makedirs(segment_output_folder, exist_ok=True)
        
        with open(segment_output_folder + '/neuron_inf.pickle', 'wb') as f:
            pickle.dump(neuron_inf_segment, f)
        
        spikes_in_segment.to_csv(segment_output_folder + '/gt_detect_array.csv', index=False)
        
        print(f"  Segment {segment_idx} 结果已保存到: {segment_output_folder}")
        print(f"    - Neurons: {len(neuron_inf_segment)}")
        print(f"    - Spikes: {len(spikes_in_segment)}")

print("\n所有clique处理完成！")


Segment 0: 采样点范围 = [0, 6000000), 采样点数 = 6000000
Segment 1: 采样点范围 = [6000000, 12000000), 采样点数 = 6000000
Segment 2: 采样点范围 = [12000000, 18000000), 采样点数 = 6000000
Segment 3: 采样点范围 = [18000000, 24000000), 采样点数 = 6000000
Segment 4: 采样点范围 = [24000000, 30000000), 采样点数 = 6000000
Segment 5: 采样点范围 = [30000000, 36000000), 采样点数 = 6000000

开始处理 6 个segments（每段 600s）


处理 Clique 0


FileNotFoundError: [Errno 2] No such file or directory: '/media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s/clique_0/phy_folder_for_kilosort/spike_times.npy'